# 01 — Preprocesamiento de Datos

**Colaborador responsable:** A (Infraestructura y Datos)

Este notebook documenta el pipeline completo de carga, limpieza y transformación del dataset COVID-19/Influenza de la DGE.

## 0. Descripción general del dataset
El dataset COVID19MEXICO.csv descrito anteriormente, contiene registros oficiales obtenidos de la Dirección General de Epidemiología publicada como dato abierto. Cada fila representa el registro de un paciente con los síntomas de COVID-19 y afines.

* **Número de filas:** 222379, supera el límite solicitado para el proyecto

* **Número de columnas:** 42

* **Periodo de fechas cubierto:** 01-01-2025 al 31-05-2026

* **Unidad de análisis:** En el dataset cada fila representa el reporte de una persona internada con los síntomas de COVID19 y afines, obteniendo datos como el ID del registro, sexo, padecimientos, cómo fue atendido, si sobrevivió, nacionalidad, entre otros. Todo el dataset esta codificado con claves numéricas del catálogo de la SSA.

El dataset tiene **42 columnas** agrupadas por temática. Los valores numéricos son **códigos de catálogo**, no medidas directas.

**Códigos especiales transversales**

| Código | Significado |
|--------|-------------|
| `1` | Sí (variables binarias) |
| `2` | No (variables binarias) |
| `97` | No aplica |
| `98` | Se ignora / sin dato |
| `99` | No especificado |
| `9999-99-99` | Fecha simbólica, es decir que el paciente sobrevivió |

---

**Identificación**

| Columna | Descripción |
|---------|-------------|
| `ID_REGISTRO` | Folio único del caso (alfanumérico) |
| `FECHA_ACTUALIZACION` | Fecha de la última actualización del registro |
| `ORIGEN` | Unidad de vigilancia (1 = USMER, Unidades de Salud Monitoras de Enfermedades Respiratorias) |

**Demografía**

| Columna | Descripción |
|---------|-------------|
| `SEXO` | 1 = Mujer · 2 = Hombre |
| `EDAD` | Edad en años (valor real) |
| `ENTIDAD_RES / NAC / UM` | Clave INEGI del estado (01 = Ags … 32 = Zac) |
| `MUNICIPIO_RES` | Clave municipal INEGI (3 dígitos) |
| `NACIONALIDAD` | 1 = Mexicano · 2 = Extranjero |
| `EMBARAZO` | 1=Sí · 2=No · 97=No aplica (hombres) |
| `HABLA_LENGUA_INDIG` | 1=Sí · 2=No |
| `INDIGENA` | 1=Sí · 2=No |
| `MIGRANTE` | 1=Sí · 2=No |

**ENTIDAD_RES = estado donde vive el paciente** <br>
**ENTIDAD_NAC = estado donde nació** <br>
**ENTIDAD_UM = estado donde esta la unidad médica que lo atendió** <br>
**MUNICIPIO_RES = municipio de residencia** <br>

El INEGI asigna un número del 01 al 32 en orden alfabético de la siguiente manera:

<div style="width: 40%">

| Clave | Entidad federativa   |
| ----- | -------------------- |
| 01    | Aguascalientes       |
| 02    | Baja California      |
| 03    | Baja California Sur  |
| 04    | Campeche             |
| 05    | Coahuila de Zaragoza |
| 06    | Colima               |
| 07    | Chiapas              |
| 08    | Chihuahua            |
| 09    | Ciudad de México     |
| 10    | Durango              |
| 11    | Guanajuato           |
| 12    | Guerrero             |
| 13    | Hidalgo              |
| 14    | Jalisco              |
| 15    | Estado de México     |
| 16    | Michoacán            |
| 17    | Morelos              |
| 18    | Nayarit              |
| 19    | Nuevo León           |
| 20    | Oaxaca               |
| 21    | Puebla               |
| 22    | Querétaro            |
| 23    | Quintana Roo         |
| 24    | San Luis Potosí      |
| 25    | Sinaloa              |
| 26    | Sonora               |
| 27    | Tabasco              |
| 28    | Tamaulipas           |
| 29    | Tlaxcala             |
| 30    | Veracruz             |
| 31    | Yucatán              |
| 32    | Zacatecas            |
</div>

**Comorbilidades** *(1=Sí · 2=No · 98=Se ignora)*

| Columna | Descripción |
|---------|-------------|
| `DIABETES` | Diabetes mellitus |
| `EPOC` | Enfermedad Pulmonar Obstructiva Crónica |
| `ASMA` | Asma |
| `INMUSUPR` | Inmunosupresión |
| `HIPERTENSION` | Hipertensión arterial |
| `CARDIOVASCULAR` | Enfermedad cardiovascular |
| `OBESIDAD` | Obesidad |
| `RENAL_CRONICA` | Enfermedad renal crónica |
| `TABAQUISMO` | Tabaquismo activo |
| `OTRA_COM` | Otra comorbilidad |

**Desenlace clínico** *(base para construir `SEVERIDAD`)*

| Columna | Descripción |
|---------|-------------|
| `TIPO_PACIENTE` | 1 = Ambulatorio · 2 = Hospitalizado |
| `NEUMONIA` | 1=Sí · 2=No |
| `INTUBADO` | 1=Sí · 2=No · 97=No aplica |
| `UCI` | Unidad de Cuidados Intensivos: 1=Sí · 2=No · 97=No aplica |
| `FECHA_INGRESO` | Fecha de ingreso a la unidad médica |
| `FECHA_SINTOMAS` | Fecha de inicio de síntomas |
| `FECHA_DEF` | Fecha de defunción (`9999-99-99` = sobrevivió) |

**Diagnóstico / laboratorio**

| Columna | Descripción |
|---------|-------------|
| `TOMA_MUESTRA_LAB` | 1=Se tomó muestra · 2=No |
| `RESULTADO_PCR` | Código del patógeno detectado por PCR |
| `RESULTADO_PCR_COINFECCION` | Coinfección detectada |
| `TOMA_MUESTRA_ANTIGENO` | 1=Sí · 2=No |
| `RESULTADO_ANTIGENO` | Resultado prueba antigénica |
| `CLASIFICACION_FINAL_COVID` | 3=Confirmado lab · 4=No conclusivo · 5=Negativo · 6=Sospechoso · 7=No clasificado |
| `CLASIFICACION_FINAL_FLU` | Misma escala para Influenza |
| `SECTOR` | Sector de la unidad médica (IMSS, ISSSTE, SSA, etc.) |

**Variable objetivo derivada**

| `SEVERIDAD` | Descripción |
|------------|-------------|
| `0` | Leve — ambulatorio |
| `1` | Grave — hospitalizado sin UCI/intubación |
| `2` | Crítico — UCI o intubado |
| `3` | Fallecido |

## 1. Setup y configuración

Agregamos las siguientes librerías para el preprocesamiento del dataset.

| Librería | Rol |
|----------|-----|
| `pandas` | Carga y manipulación del DataFrame |
| `numpy` | Operaciones vectoriales (construcción de variable `SEVERIDAD`) |
| `matplotlib` / `seaborn` | Visualizaciones (disponibles para EDA posterior) |
| `DataLoader` | Carga y validación del CSV crudo |
| `DataPreprocessor` | Pipeline de limpieza y transformación |
| `constants` | Constantes centralizadas: rutas, semilla, nombres de columnas |

In [35]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.data.loader import DataLoader
from src.data.preprocessor import DataPreprocessor
from src.utils.constants import (
    RANDOM_STATE, SEVERIDAD_MAP, COLS_COMORBILIDADES,
    DATA_RAW_PATH, CSV_FILENAME,
)

## 2. Carga del dataset crudo

In [36]:
loader = DataLoader()
df_raw = loader.load()
loader.descripcion_general()

El dataset se cargó correctamente: **222,379 registros y 42 columnas**.
Todas las columnas tienen nombre sin espacios en blanco.
El tipo predominante es `int64` (variables de catálogo codificadas numéricamente), y la única variable continua real es `EDAD`.

## 3. Análisis de calidad de datos

Hacemos la previsualización de los valores faltantes, duplicados y códigos especiales (97, 98, 99) a través de una tabla mostrando el porcentaje de dichos datos.

In [37]:
df_calidad = loader.resumen_calidad()
df_calidad

,columna,tipo,nulos,pct_nulos,codigos_especiales,pct_especiales,valores_unicos
0,FECHA_ACTUALIZACION,str,0,0.000000,0,0.000000,1
1,ID_REGISTRO,str,0,0.000000,0,0.000000,222379
2,ORIGEN,int64,0,0.000000,0,0.000000,1
3,SECTOR,int64,0,0.000000,0,0.000000,12
4,ENTIDAD_UM,int64,0,0.000000,0,0.000000,32
5,SEXO,int64,0,0.000000,0,0.000000,2
6,ENTIDAD_NAC,int64,0,0.000000,935,0.420000,33
7,ENTIDAD_RES,int64,0,0.000000,0,0.000000,32
8,MUNICIPIO_RES,int64,0,0.000000,1766,0.790000,435
9,TIPO_PACIENTE,int64,0,0.000000,0,0.000000,2


Como podemos notar, a diferencia de otros datasets, este dataset esta bastante limpio, las únicas columnas con mayor porcentaje de **códigos especiales** (97/98/99) son las clínicas:
`INTUBADO`, `UCI`, `EMBARAZO` y esto es lógico puesto que aplican solo a subgrupos de pacientes (hospitalizados, mujeres, entre otros).
Los nulos son cero en el CSV crudo porque los valores faltantes están codificados como 97/98/99, no como `NaN`.

In [44]:
loader.detectar_duplicados()

Registros duplicados: 0 (0.00%)


0

**El dataset tiene 0 registros duplicados** es decir que cada fila corresponde a un caso único identificado por `ID_REGISTRO`, por lo tando no se requiere eliminación de duplicados en esta etapa.

### 3.1 Estrategia de manejo de valores faltantes

Los **códigos especiales 97, 98 y 99** del catálogo de la DGE no son valores reales sino que estan codificados con los siguientes significados:

| Código | Significado |
|--------|-------------|
| `97` | No aplica (ej: `INTUBADO` en paciente ambulatorio) |
| `98` | Se ignora / sin dato |
| `99` | No especificado |

**Decisión**: reemplazamos 97/98/99 por `NaN` en columnas clínicas de comorbilidades y demográficas binarias. **Y no eliminamos filas** porque puede que a pesar de no contar con esa información en específico, se pueden utilizar otros campos para el análisis, además de que los modelos de árbol (XGBoost, Random Forest) manejan `NaN` nativamente.

**Fecha de defunción (`FECHA_DEF`)**: el valor `9999-99-99` indica que el paciente **sobrevivió**. Por lo que decidimos convertirlo a `NaT` para que la columna sea una fecha válida y practica utilizando la función `notna()` para filtrar a las personas fallecidas y `isna()` para filtrar a las personas sobrevivientes.

## 4. Creación de la variable SEVERIDAD

Creamos una nueva variable objetivo ordinal con 4 niveles:
- 0: Leve (ambulatorio)
- 1: Grave (hospitalizado)
- 2: Crítico (UCI/intubado)
- 3: Fallecido
Esto porque el dataset no cuenta con una columna que indique si el paciente tuvo un historial clínico grave, tomamos en cuenta las variables TIPO_PACIENTE, UCI, INTUBADO, FECHA_DEF. 

In [39]:
preprocessor = DataPreprocessor()
df_clean = preprocessor.fit_transform(df_raw)

print(f"Dimensiones originales : {df_raw.shape[0]:,} filas × {df_raw.shape[1]} columnas")
print(f"Dimensiones limpias    : {df_clean.shape[0]:,} filas × {df_clean.shape[1]} columnas")
print(f"Columnas añadidas      : SEVERIDAD, EDAD_SCALED")

Dimensiones originales : 222,379 filas × 42 columnas
Dimensiones limpias    : 222,379 filas × 44 columnas
Columnas añadidas      : SEVERIDAD, EDAD_SCALED


El código revisa las 4 condiciones, todo paciente esta inicializado en 0 inicialmente, después checa si fue hospitalizado y si sí se actualiza a 1 punto, si estuvo en UCI o si estuvo intubado se actualiza a 2 puntos, si el paciente falleció se actualiza a 3 puntos.

Hacemos 4 transformaciones secuenciales sobre los registros:
1. Códigos 97/98/99 a `NaN` en columnas clínicas, comorbilidades y demográficas.
2. `FECHA_DEF` con valor `9999-99-99` lo pasamos a `NaT` para indicar sobrevivientes.
3. Creamos la variable `SEVERIDAD` para tener una mejor lógica de prioridad descendente.
4. Encoding binario y estandarización de `EDAD`

Añadimos 2 columnas más (`SEVERIDAD`, `EDAD_SCALED`): **42 a 44 columnas**.

In [40]:
from src.utils.constants import SEVERIDAD_MAP

dist = df_clean["SEVERIDAD"].value_counts().sort_index()
total = dist.sum()

print("Distribución de SEVERIDAD\n")
print(f"{'Nivel':<5} {'Descripción':<30} {'Casos':>8} {'%':>7}")
print("-" * 55)
for nivel, desc in SEVERIDAD_MAP.items():
    n = dist.get(nivel, 0)
    print(f"  {nivel}    {desc:<30} {n:>8,} {n/total*100:>6.1f}%")
print("-" * 55)
print(f"{'Total':<36} {total:>8,} {'100.0%':>7}")

Distribución de SEVERIDAD

Nivel Descripción                       Casos       %
-------------------------------------------------------
  0    Leve (ambulatorio)              116,190   52.2%
  1    Grave (hospitalizado)            93,828   42.2%
  2    Crítico (UCI/intubado)            4,114    1.8%
  3    Fallecido                         8,247    3.7%
-------------------------------------------------------
Total                                 222,379  100.0%


La distribución refleja la **pirámide de severidad** esperada en un sistema de vigilancia centinela:
la mayoría de casos son ambulatorios (leves), con una fracción menor de hospitalizados, y aún menor de casos críticos y fallecidos.
El desbalance de clases deberá manejarse en el modelado (ej. `class_weight`, oversampling o métricas ajustadas).

## 5. Transformaciones aplicadas

### 5.1 Encoding de categóricas

In [41]:
from src.utils.constants import COLS_COMORBILIDADES

binary_si_no = COLS_COMORBILIDADES + [
    "NEUMONIA", "TOMA_MUESTRA_LAB", "INTUBADO", "UCI",
    "EMBARAZO", "HABLA_LENGUA_INDIG", "INDIGENA", "MIGRANTE",
    "OTRO_CASO", "NACIONALIDAD",
]

print("Reglas de encoding aplicadas:")
print(f"  • {len(binary_si_no)} columnas binarias (1=Sí/2=No)  →  1 / 0")
print( "  • TIPO_PACIENTE  (1=ambulatorio / 2=hospitalizado)  →  0 / 1")
print( "  • SEXO           (1=Mujer / 2=Hombre)               →  0 / 1")
print("\nValores únicos post-encoding (sin NaN):")
for col in binary_si_no + ["TIPO_PACIENTE", "SEXO"]:
    if col in df_clean.columns:
        vals = sorted(df_clean[col].dropna().unique().tolist())
        print(f"  {col:<30} {vals}")

Reglas de encoding aplicadas:
  • 20 columnas binarias (1=Sí/2=No)  →  1 / 0
  • TIPO_PACIENTE  (1=ambulatorio / 2=hospitalizado)  →  0 / 1
  • SEXO           (1=Mujer / 2=Hombre)               →  0 / 1

Valores únicos post-encoding (sin NaN):
  DIABETES                       [0.0, 1.0]
  EPOC                           [0.0, 1.0]
  ASMA                           [0.0, 1.0]
  INMUSUPR                       [0.0, 1.0]
  HIPERTENSION                   [0.0, 1.0]
  OTRA_COM                       [0.0, 1.0]
  CARDIOVASCULAR                 [0.0, 1.0]
  OBESIDAD                       [0.0, 1.0]
  RENAL_CRONICA                  [0.0, 1.0]
  TABAQUISMO                     [0.0, 1.0]
  NEUMONIA                       [0, 1]
  TOMA_MUESTRA_LAB               [0, 1]
  INTUBADO                       [0.0, 1.0]
  UCI                            [0.0, 1.0]
  EMBARAZO                       [0.0, 1.0]
  HABLA_LENGUA_INDIG             [0.0, 1.0]
  INDIGENA                       [0.0, 1.0]
  MIGRANTE      

Todos los valores únicos post-encoding son `[0, 1]` (o `NaN` donde aplica).
El encoding es consistente: **1 = presencia / condición positiva**, **0 = ausencia**.
Las variables con `NaN` conservan su falta de información; no se imputa en esta etapa.

### 5.2 Escalado de numéricas

In [ ]:
print("Escalado aplicado: StandardScaler sobre EDAD a EDAD_SCALED")
print("La columna EDAD original se conserva para interpretabilidad en el EDA.\n")

stats = df_clean[["EDAD", "EDAD_SCALED"]].describe().round(3)
print(stats.to_string())

Escalado aplicado: StandardScaler sobre EDAD → EDAD_SCALED
La columna EDAD original se conserva para interpretabilidad en el EDA.

             EDAD  EDAD_SCALED
count  222379.000   222379.000
mean       35.102       -0.000
std        26.444        1.000
min         0.000       -1.327
25%         9.000       -0.987
50%        32.000       -0.117
75%        56.000        0.790
max       121.000        3.248


`EDAD_SCALED` tiene media ≈ 0 y desviación estándar ≈ 1, confirmando la estandarización correcta.
La columna `EDAD` original se mantiene para que el EDA sea interpretable en años reales.

## 6. Exportación de datos limpios

In [43]:
preprocessor.exportar(df_clean)

Dataset limpio exportado a: /home/zyann28/Documentos/DatosYmineria/proyecto-final/data/COVID19MEXICO_clean.csv
Dimensiones: 222,379 filas × 44 columnas


El archivo `COVID19MEXICO_clean.csv` queda listo en `data/` con **222,379 filas × 44 columnas**.
Este es el punto de entrada para el notebook de EDA y para los pipelines de modelado.

## 7. Resumen del preprocesamiento

| Etapa | Descripción | Resultado |
|-------|-------------|-----------|
| **Carga** | CSV con codificación `latin-1` vía `DataLoader` | 222,379 filas × 42 columnas |
| **Duplicados** | Detección con `detectar_duplicados()` | 0 duplicados (0.00 %) |
| **Valores faltantes** | Códigos 97/98/99 → `NaN` en columnas clínicas, comorbilidades y demográficas. `FECHA_DEF` centinela `9999-99-99` → `NaT` | Sin eliminación de filas |
| **Variable objetivo** | `SEVERIDAD` ordinal (0–3) derivada de `TIPO_PACIENTE`, `UCI`, `INTUBADO` y `FECHA_DEF` con prioridad descendente | +1 columna |
| **Encoding** | Binarias 1/2 → 1/0 · `TIPO_PACIENTE` 1/2 → 0/1 · `SEXO` 1/2 → 0/1 | 22 columnas recodificadas |
| **Escalado** | `StandardScaler` sobre `EDAD` → `EDAD_SCALED` (columna original conservada) | +1 columna |
| **Exportación** | `COVID19MEXICO_clean.csv` guardado en `data/` | 222,379 filas × 44 columnas |